<a href="https://colab.research.google.com/github/Niyatigupta21/India_Tech_Market_Analytics/blob/main/India_Tech_Market_Dataset_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#---------------------------------------------------------------
**India Tech Hiring Intelligence 2025-2026**
#---------------------------------------------------------------

Reads the raw uploaded dataset and produces a fully cleaned, documented
analytical dataset. No row is silently dropped except exact full-row
duplicates. No missing salary is coerced to 0. An audit trail (audit_log)
records every major transformation with counts, for the README / data
quality report.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import json


pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

# ---------------------------------------------------------------
# 1. LOAD DATA
# ---------------------------------------------------------------

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving indian-job-market-dataset-2025.xlsx to indian-job-market-dataset-2025 (1).xlsx


In [ ]:
df = pd.read_excel("indian-job-market-dataset-2025.xlsx")

In [ ]:
df.shape

(97929, 17)

In [ ]:
df.head()

,title,jobId,currency,jobUploaded,companyName,tagsAndSkills,experience,salary,location,companyId,ReviewsCount,AggregateRating,jobDescription,minimumSalary,maximumSalary,minimumExperience,maximumExperience
0,Sr. HR Recruiter (NON IT),270925008041,INR,6 Days Ago,Orion,"Communication,Manpower,Staffing,Convincing Power,Hiring,Recruitment,SR,Communication skills",2-4 Yrs,2-4 Lacs PA,Kolkata(Chinar Park),645563,NaN,NaN,Preferred candidate profile . .,200000.0,400000.0,2.0,4.0
1,Fire And Safety Officer,270925007584,INR,6 Days Ago,"Apollo Hospitals International Limited, Ahmedabad","Safety Officer Activities,Fire Protection,Fire Safety,Fire Engineering,Fire Prevention,Safety Ma...",6-11 Yrs,3-5 Lacs PA,"Gandhinagar, Ahmedabad",14072,5162.0,4.0,"Ensure active Fire Protection System,such as Fire Hydrant system,Fire extinguishers,Fire Alarm s...",300000.0,500000.0,6.0,11.0
2,Opening For Performance Marketing - Chennai,270925007492,INR,6 Days Ago,TVS Credit Services Ltd,"Performance Marketing,User Acquisition,growth marketing,Paid Marketing,Acquisition,Performance,U...",12-18 Yrs,Not disclosed,Chennai,1324750,2892.0,4.2,MBA Marketing (preferred Tier II or III B- School). <br><br>Experience <br><br> 12+ years exp in...,0.0,0.0,12.0,18.0
3,Medical Billing Executive,270925007443,INR,6 Days Ago,GNR Global Services,"Fluent English,Spoken English,Good English Communication,Medical,Medical billing,Billing,Fluent,...",0-3 Yrs,"70,000-2 Lacs PA","Mohali, Chandigarh, Kharar, Zirakpur",123804403,NaN,NaN,Job Title-Medical Billing Executive\nLocation-Mohali\nSalary-20-22k ctc\n\nBenefits:\nCab facili...,70000.0,200000.0,0.0,3.0
4,Senior Group Product Manager - CNS Therapy,270925007430,INR,6 Days Ago,Cadila Pharmaceuticals,"Product Marketing,CNS,Product Management,Nephrology,Group Product Management,Brand Marketing,Neu...",5-10 Yrs,8-18 Lacs PA,Ahmedabad,14957,2134.0,3.4,Principal Tasks & Responsibilities : (Please write all the major jobs that the employee is requi...,800000.0,1800000.0,5.0,10.0


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97929 entries, 0 to 97928
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   title              97929 non-null  object 
 1   jobId              97929 non-null  int64  
 2   currency           97929 non-null  object 
 3   jobUploaded        97929 non-null  object 
 4   companyName        97925 non-null  object 
 5   tagsAndSkills      97358 non-null  object 
 6   experience         95824 non-null  object 
 7   salary             97929 non-null  object 
 8   location           97929 non-null  object 
 9   companyId          97929 non-null  int64  
 10  ReviewsCount       62677 non-null  float64
 11  AggregateRating    62677 non-null  float64
 12  jobDescription     97929 non-null  object 
 13  minimumSalary      97358 non-null  float64
 14  maximumSalary      97358 non-null  float64
 15  minimumExperience  97358 non-null  float64
 16  maximumExperience  973

In [ ]:
df.isnull().sum()

,0
title,0
jobId,0
currency,0
jobUploaded,0
companyName,4
tagsAndSkills,571
experience,2105
salary,0
location,0
companyId,0


# ---------------------------------------------------------------
# 2. DEDUPLICATE (exact full-row duplicates only)
# ---------------------------------------------------------------

In [ ]:
before = len(df)

df = df.drop_duplicates(keep="first").reset_index(drop=True)

after = len(df)

print(f"Dropped {before - after} exact full-row duplicate rows")
print(f"{after} rows remain")

# Give every surviving row a clean internal ID
df["row_id"] = df.index + 1

Dropped 247 exact full-row duplicate rows
97682 rows remain


# ---------------------------------------------------------------
# 3. SALARY CLEANING  (the "missing salary != 0" fix)
# --------------------------------------------------------------

In [ ]:
def parse_salary(row):
    """
    Returns:
    (min_salary_clean, max_salary_clean, salary_status)

    salary_status:
    Disclosed / Not Disclosed / Unpaid / Stipend
    """

    raw = str(row["salary"]).strip()

    # Unpaid role
    if raw == "Unpaid":
        return 0.0, 0.0, "Unpaid"

    # Salary not disclosed
    if raw == "Not disclosed":
        return np.nan, np.nan, "Not Disclosed"

    # Monthly salary/stipend → annual salary
    if "/month" in raw:
        m = re.search(r"([\d,]+)", raw)

        if m:
            monthly = float(m.group(1).replace(",", ""))
            annual = monthly * 12
            return annual, annual, "Stipend"

        return np.nan, np.nan, "Stipend"

    # For normal disclosed salaries,
    # use the dataset's existing minimum and maximum salary
    mn = row["minimumSalary"]
    mx = row["maximumSalary"]

    if pd.isna(mn) or pd.isna(mx):
        return np.nan, np.nan, "Not Disclosed"

    return float(mn), float(mx), "Disclosed"


# Apply salary parser
parsed = df.apply(parse_salary, axis=1, result_type="expand")

parsed.columns = [
    "minimum_salary_clean",
    "maximum_salary_clean",
    "salary_status"
]

# Add cleaned salary columns to dataframe
df = pd.concat([df, parsed], axis=1)


# Calculate salary midpoint
df["salary_midpoint"] = (
    df["minimum_salary_clean"] +
    df["maximum_salary_clean"]
) / 2


# Create useful flags
df["is_unpaid_internship"] = (
    df["salary_status"] == "Unpaid"
).astype(int)

df["is_stipend_role"] = (
    df["salary_status"] == "Stipend"
).astype(int)

In [ ]:
# Currency flag
df["currency_note"] = np.where(
    df["currency"] != "INR",
    "Non-INR - exclude from ₹ salary aggregates",
    "INR"
)

In [ ]:
df["salary_status"].value_counts()

,count
salary_status,
Not Disclosed,63920
Disclosed,33191
Unpaid,333
Stipend,238


In [ ]:
df[
    ["salary", "minimum_salary_clean", "maximum_salary_clean", "salary_status", "salary_midpoint"]
].head(15)

,salary,minimum_salary_clean,maximum_salary_clean,salary_status,salary_midpoint
0,2-4 Lacs PA,200000.0,400000.0,Disclosed,300000.0
1,3-5 Lacs PA,300000.0,500000.0,Disclosed,400000.0
2,Not disclosed,NaN,NaN,Not Disclosed,NaN
3,"70,000-2 Lacs PA",70000.0,200000.0,Disclosed,135000.0
4,8-18 Lacs PA,800000.0,1800000.0,Disclosed,1300000.0
5,3.5-6 Lacs PA,350000.0,600000.0,Disclosed,475000.0
6,Not disclosed,NaN,NaN,Not Disclosed,NaN
7,Not disclosed,NaN,NaN,Not Disclosed,NaN
8,Not disclosed,NaN,NaN,Not Disclosed,NaN
9,Not disclosed,NaN,NaN,Not Disclosed,NaN


# ---------------------------------------------------------------
# 4. EXPERIENCE CLEANING
# ---------------------------------------------------------------

In [ ]:
# 4. EXPERIENCE CLEANING

# Use numeric minimum and maximum experience as the main source
df["minimum_experience_clean"] = df["minimumExperience"]
df["maximum_experience_clean"] = df["maximumExperience"]

# Calculate experience midpoint
df["experience_midpoint"] = (
    df["minimum_experience_clean"] +
    df["maximum_experience_clean"]
) / 2

In [ ]:
# Fresher-friendly rule:
# minimum experience <= 1 year

df["is_fresher_friendly"] = np.where(
    df["minimum_experience_clean"].notna() &
    (df["minimum_experience_clean"] <= 1),
    1,
    0
)

In [ ]:
print("Missing experience:",
      df["minimum_experience_clean"].isna().sum())

print("Fresher-friendly jobs:",
      df["is_fresher_friendly"].sum())

print("Total jobs:",
      len(df))

Missing experience: 571
Fresher-friendly jobs: 27674
Total jobs: 97682


# ---------------------------------------------------------------
# 5. LOCATION CLEANING
# ---------------------------------------------------------------

In [ ]:
def clean_location(loc):

    loc = str(loc).strip()

    # Detect remote jobs
    is_remote = 1 if re.search(r"\bremote\b", loc, re.I) else 0

    # Detect hybrid jobs
    is_hybrid = 1 if loc.lower().startswith("hybrid") else 0

    # Remove "Hybrid - " prefix
    loc2 = re.sub(
        r"^hybrid\s*-\s*",
        "",
        loc,
        flags=re.I
    )

    # Remove area details inside brackets
    # Example: Kolkata(Chinar Park) → Kolkata
    loc2 = re.sub(r"\(.*?\)", "", loc2).strip()

    # Detect multiple cities
    is_multi = 1 if "," in loc2 else 0

    # Take first city
    first_city = loc2.split(",")[0].strip()

    # Standardize Remote
    if is_remote and first_city.lower() == "remote":
        first_city = "Remote"

    # Determine work type
    work_type = (
        "Remote" if is_remote
        else ("Hybrid" if is_hybrid else "Onsite")
    )

    return pd.Series([
        first_city,
        work_type,
        is_remote,
        is_multi
    ])


df[
    [
        "city",
        "work_location_type",
        "is_remote",
        "is_multi_city"
    ]
] = df["location"].apply(clean_location)

In [ ]:
CITY_STATE_MAP = {
    "Bengaluru": "Karnataka",
    "Bangalore": "Karnataka",
    "Mysuru": "Karnataka",
    "Mangalore": "Karnataka",

    "Hyderabad": "Telangana",
    "Warangal": "Telangana",

    "Pune": "Maharashtra",
    "Mumbai": "Maharashtra",
    "Navi Mumbai": "Maharashtra",
    "Thane": "Maharashtra",
    "Nagpur": "Maharashtra",
    "Nashik": "Maharashtra",
    "Aurangabad": "Maharashtra",
    "Mumbai Suburban": "Maharashtra",

    "Chennai": "Tamil Nadu",
    "Coimbatore": "Tamil Nadu",
    "Madurai": "Tamil Nadu",
    "Trichy": "Tamil Nadu",

    "Gurugram": "Haryana",
    "Gurgaon": "Haryana",
    "Faridabad": "Haryana",

    "Noida": "Uttar Pradesh",
    "Lucknow": "Uttar Pradesh",
    "Kanpur": "Uttar Pradesh",
    "Ghaziabad": "Uttar Pradesh",
    "Agra": "Uttar Pradesh",
    "Varanasi": "Uttar Pradesh",

    "Ahmedabad": "Gujarat",
    "Surat": "Gujarat",
    "Vadodara": "Gujarat",
    "Gandhinagar": "Gujarat",
    "Rajkot": "Gujarat",

    "Kolkata": "West Bengal",
    "Howrah": "West Bengal",

    "Jaipur": "Rajasthan",
    "Jodhpur": "Rajasthan",
    "Udaipur": "Rajasthan",

    "Kochi": "Kerala",
    "Thiruvananthapuram": "Kerala",
    "Kozhikode": "Kerala",
    "Ernakulam": "Kerala",

    "Chandigarh": "Chandigarh",
    "Mohali": "Punjab",
    "Ludhiana": "Punjab",
    "Amritsar": "Punjab",

    "Indore": "Madhya Pradesh",
    "Bhopal": "Madhya Pradesh",
    "Gwalior": "Madhya Pradesh",

    "Delhi": "Delhi",
    "New Delhi": "Delhi",

    "Bhubaneswar": "Odisha",
    "Cuttack": "Odisha",

    "Guwahati": "Assam",
    "Patna": "Bihar",
    "Ranchi": "Jharkhand",
    "Dehradun": "Uttarakhand",
    "Raipur": "Chhattisgarh",

    "Visakhapatnam": "Andhra Pradesh",
    "Vijayawada": "Andhra Pradesh",

    "Remote": "Not Applicable (Remote)"
}

df["state"] = (
    df["city"]
    .map(CITY_STATE_MAP)
    .fillna("Other/Unmapped")
)

In [ ]:
print("Rows:", len(df))

print(
    "Remote jobs:",
    df["is_remote"].sum()
)

print(
    "Hybrid jobs:",
    (df["work_location_type"] == "Hybrid").sum()
)

print(
    "Multi-city jobs:",
    df["is_multi_city"].sum()
)

print(
    "Other/Unmapped cities:",
    (df["state"] == "Other/Unmapped").sum()
)

print("\nTop 20 cities:")
print(df["city"].value_counts().head(20))

print("\nWork location type:")
print(df["work_location_type"].value_counts())

Rows: 97682
Remote jobs: 1958
Hybrid jobs: 5753
Multi-city jobs: 19240
Other/Unmapped cities: 11234

Top 20 cities:
city
Bengaluru          19427
Hyderabad          12333
Pune                9378
Mumbai              6633
Chennai             6437
Gurugram            5259
Noida               4982
Ahmedabad           2436
Kolkata             1962
Remote              1952
Navi Mumbai         1701
Thane               1029
Kochi                980
Jaipur               968
Mumbai Suburban      787
Coimbatore           750
Chandigarh           625
Mohali               616
Vadodara             590
Nagpur               584
Name: count, dtype: int64

Work location type:
work_location_type
Onsite    89971
Hybrid     5753
Remote     1958
Name: count, dtype: int64


#---------------------------------------------------------------
# 6. DATA QUALITY CHECK
# ---------------------------------------------------------------

In [ ]:
print("Dataset shape:", df.shape)

missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
)

missing_pct = (
    (df.isna().mean() * 100)
      .sort_values(ascending=False)
      .round(2)
)

missing_report = pd.DataFrame({
    "missing_count": missing,
    "missing_percentage": missing_pct
})

print("\nMissing value report:")
print(missing_report)

Dataset shape: (97682, 34)

Missing value report:
                          missing_count  missing_percentage
salary_midpoint                   63920               65.44
maximum_salary_clean              63920               65.44
minimum_salary_clean              63920               65.44
AggregateRating                   35165               36.00
ReviewsCount                      35165               36.00
experience                         2102                2.15
minimum_experience_clean            571                0.58
minimumSalary                       571                0.58
maximumSalary                       571                0.58
tagsAndSkills                       571                0.58
minimumExperience                   571                0.58
maximumExperience                   571                0.58
maximum_experience_clean            571                0.58
experience_midpoint                 571                0.58
companyName                           4           

In [ ]:
important_cols = [
    "title",
    "companyName",
    "tagsAndSkills",
    "location",
    "minimum_salary_clean",
    "maximum_salary_clean",
    "minimum_experience_clean",
    "maximum_experience_clean",
    "city",
    "state"
]

print("\nImportant columns:")
print(missing_report.loc[important_cols])


Important columns:
                          missing_count  missing_percentage
title                                 0                0.00
companyName                           4                0.00
tagsAndSkills                       571                0.58
location                              0                0.00
minimum_salary_clean              63920               65.44
maximum_salary_clean              63920               65.44
minimum_experience_clean            571                0.58
maximum_experience_clean            571                0.58
city                                  0                0.00
state                                 0                0.00


In [ ]:
print("\nSalary status:")
print(df["salary_status"].value_counts(dropna=False))


Salary status:
salary_status
Not Disclosed    63920
Disclosed        33191
Unpaid             333
Stipend            238
Name: count, dtype: int64


In [ ]:
print("\nExperience availability:")
print(
    df["minimum_experience_clean"]
    .isna()
    .value_counts()
)


Experience availability:
minimum_experience_clean
False    97111
True       571
Name: count, dtype: int64


# ---------------------------------------------------------------
# 7. JOB RECENCY (jobUploaded) -- explicitly NOT used for trend claims
#---------------------------------------------------------------


In [ ]:
def recency_bucket(val):

    v = str(val).strip()

    if v.startswith("Starts"):
        return "Future Start / Internship"

    if v in ["Just Now", "Today", "Few Hours Ago"]:
        return "Today"

    if re.match(r"^[1-6] Days? Ago$", v):
        return "This Week (1-6 Days Ago)"

    if re.match(r"^(7|8|9|10) Days Ago$", v):
        return "1-2 Weeks Ago"

    if re.match(r"^(1[1-9]|20) Days Ago$", v):
        return "2-3 Weeks Ago"

    return "Other/Unparsed"


df["job_recency_bucket"] = df["jobUploaded"].apply(recency_bucket)

df["is_future_start"] = (
    df["job_recency_bucket"] == "Future Start / Internship"
).astype(int)

In [ ]:
def parse_start_date(val):

    v = str(val)

    m = re.search(
        r"Starts\s*:\s*(\d{1,2})\w{0,2}\s+(\w+)'\s*(\d{2})",
        v
    )

    if not m:
        return pd.NaT

    day, month, year = m.groups()

    return pd.to_datetime(
        f"{day} {month} 20{year}",
        format="%d %b %Y",
        errors="coerce"
    )


df["start_date"] = df["jobUploaded"].apply(parse_start_date)

In [ ]:
print("Recency buckets:")
print(df["job_recency_bucket"].value_counts())

print("\nFuture-start jobs:")
print(df["is_future_start"].sum())

print("\nParsed start dates:")
print(df["start_date"].notna().sum())

Recency buckets:
job_recency_bucket
1-2 Weeks Ago                46324
This Week (1-6 Days Ago)     39816
Today                        10969
Future Start / Internship      571
2-3 Weeks Ago                    2
Name: count, dtype: int64

Future-start jobs:
571

Parsed start dates:
33


# ---------------------------------------------------------------
# 8. COMPANY NAME MISSING
# ---------------------------------------------------------------

In [ ]:
print("Missing company names before:",
      df["companyName"].isna().sum())

df["companyName"] = (
    df["companyName"]
    .fillna("Unknown Company")
)

Missing company names before: 0


# ---------------------------------------------------------------
# 9. SKILLS FIELD
# ---------------------------------------------------------------

In [ ]:
df["tagsAndSkills"] = df["tagsAndSkills"].fillna("")

df["skill_count"] = df["tagsAndSkills"].apply(
    lambda s: len([
        t for t in str(s).split(",")
        if t.strip()
    ])
)

print("Jobs with zero listed skills:",
      (df["skill_count"] == 0).sum())

Jobs with zero listed skills: 571


#---------------------------------------------------------------
# 10. RATINGS - never coerce missing to 0
# ---------------------------------------------------------------


In [ ]:
print("Missing ratings:",
      df["AggregateRating"].isna().sum())

print("Missing review counts:",
      df["ReviewsCount"].isna().sum())

In [ ]:
print("\nFinal shape:", df.shape)

print("\nNew columns:")
print([
    "companyName",
    "skill_count"
])


Final shape: (97682, 38)

New columns:
['companyName', 'skill_count']


#---------------------------------------------------------------
# TECH JOB CLASSIFICATION
# ---------------------------------------------------------------

# ---------------------------------------------------------------
# TIER 1: TITLE-BASED TECH ROLE RULES
# ---------------------------------------------------------------

In [ ]:
TITLE_CATEGORY_RULES = [
    ("Data Analytics",
     r"\bdata analyst\b"),

    ("Data Science / ML",
     r"\bdata scientist\b|\bmachine learning\b|\bml engineer\b"),

    ("AI / GenAI",
     r"\bgenai\b|\bgenerative ai\b|\bai engineer\b|\bai/ml\b|\bai ml\b"),

    ("Data Engineering",
     r"\bdata engineer\b"),

    ("Cloud / DevOps",
     r"\bdevops\b|\bcloud engineer\b|\bcloud architect\b|\bsite reliability\b|\bsre\b"),

    ("Cybersecurity",
     r"\bcyber ?security\b|\bsecurity engineer\b|\bpenetration test"),

    ("QA / Testing",
     r"\bqa\b|\bquality assurance\b|\btest engineer\b|\btester\b|\bautomation testing\b"),

    ("Web Development",
     r"\bfrontend\b|\bfront end\b|\bbackend\b|\bback end\b|\bweb developer\b"),

    ("Software Development",
     r"\bsoftware engineer\b|\bsoftware developer\b|\bsde\b|\bfull ?stack\b|\bjava developer\b|\b\.net developer\b|\bpython developer\b"),

    ("IT Support",
     r"\bit support\b|\btechnical support\b|\bapplication support\b|\bhelp ?desk\b"),
]


# ---------------------------------------------------------------
# TECH SKILL WHITELIST
# ---------------------------------------------------------------

In [ ]:
TECH_SKILL_WHITELIST = {
    "python",
    "sql",
    "java",
    "javascript",
    "c#",
    "c++",
    "aws",
    "azure",
    "microsoft azure",
    "gcp",
    "react.js",
    "react",
    "angular",
    "spring boot",
    "kubernetes",
    "docker",
    "power bi",
    "tableau",
    "advanced excel",
    "machine learning",
    "tensorflow",
    "pytorch",
    "hadoop",
    "spark",
    "pyspark",
    "hive",
    "airflow",
    "terraform",
    "jenkins",
    "ci/cd",
    "continuous integration",
    "html",
    "css",
    "git",
    "github",
    "linux",
    "microservices",
    "natural language processing",
    "rest",
    "node.js",
    "scala",
    "etl",
    "data engineering",
    "data science",
    "data analysis",
    "data analytics",
    "deep learning",
    "artificial intelligence",
    "generative ai",
    "llm",
    "devops",
    "automation testing",
    "software testing",
    "networking",
    "database",
    "mysql",
    "postgresql",
    "mongodb",
    "data modeling",
    "business intelligence",
    "bi",
    "data visualization"
}


# ---------------------------------------------------------------
# BORDERLINE ROLES
# These are not automatically treated as tech.
# Their skills decide whether they are tech roles.
# ---------------------------------------------------------------

In [ ]:
BORDERLINE_TITLE_RULES = {
    r"\bbusiness analyst\b|\bproduct analyst\b":
        "Business / Product Analytics",

    r"\bmis executive\b":
        "Business / Product Analytics",

    r"\berp\b|\bsap\b":
        "Other Tech",
}


# ---------------------------------------------------------------
# DEFINITELY NON-TECH TITLES
# ---------------------------------------------------------------

In [ ]:
NO_TITLE_RULES = [
    r"\bit recruiter\b|\bus it recruiter\b|\btechnical recruiter\b",
    r"\bdata entry\b",
]



# ---------------------------------------------------------------
# INFER CATEGORY FROM SKILLS
# ---------------------------------------------------------------

In [ ]:
def infer_category_from_skills(skill_hits):

    mapping = [

        (
            "Data Engineering",
            {
                "pyspark",
                "hadoop",
                "spark",
                "hive",
                "airflow",
                "etl",
                "data engineering",
                "data modeling",
                "scala"
            }
        ),

        (
            "Data Science / ML",
            {
                "machine learning",
                "tensorflow",
                "pytorch",
                "deep learning",
                "data science",
                "natural language processing",
                "artificial intelligence"
            }
        ),

        (
            "AI / GenAI",
            {
                "generative ai",
                "llm"
            }
        ),

        (
            "Cloud / DevOps",
            {
                "kubernetes",
                "docker",
                "terraform",
                "jenkins",
                "ci/cd",
                "continuous integration",
                "devops",
                "aws",
                "azure",
                "microsoft azure",
                "gcp"
            }
        ),

        (
            "Data Analytics",
            {
                "power bi",
                "tableau",
                "data analysis",
                "data analytics",
                "data visualization",
                "business intelligence",
                "bi",
                "advanced excel"
            }
        ),

        (
            "Software Development",
            {
                "java",
                "javascript",
                "c#",
                "c++",
                "spring boot",
                "microservices",
                "html",
                "css",
                "react.js",
                "react",
                "node.js",
                "git",
                "github"
            }
        ),

        (
            "QA / Testing",
            {
                "software testing",
                "automation testing"
            }
        ),

        (
            "Cybersecurity",
            set()
        )
    ]

    for category, keys in mapping:

        if skill_hits & keys:
            return category

    return "Other Tech"


# ---------------------------------------------------------------
# MAIN CLASSIFICATION FUNCTION
# ---------------------------------------------------------------


In [ ]:
def classify(row):

    title = str(row["title"]).lower()

    skills_raw = str(row["tagsAndSkills"]).lower()

    skill_set = {
        s.strip()
        for s in skills_raw.split(",")
        if s.strip()
    }

    skill_hits = skill_set & TECH_SKILL_WHITELIST

    # 1. Explicit non-tech titles
    for pattern in NO_TITLE_RULES:

        if re.search(pattern, title):
            return (
                "No",
                "Other/Non-Tech",
                f"title_exclusion:{pattern}"
            )

    # 2. Strong tech title match
    for category, pattern in TITLE_CATEGORY_RULES:

        if re.search(pattern, title):
            return (
                "Yes",
                category,
                f"title_match:{pattern}"
            )

    # 3. Borderline roles
    for pattern, category in BORDERLINE_TITLE_RULES.items():

        if re.search(pattern, title):

            if len(skill_hits) >= 1:
                return (
                    "Yes",
                    category,
                    f"borderline_override:{pattern}+skills({len(skill_hits)})"
                )

            else:
                return (
                    "Borderline",
                    category,
                    f"borderline_no_tech_skill:{pattern}"
                )

    # 4. Generic title + at least 2 technical skills
    if len(skill_hits) >= 2:

        category = infer_category_from_skills(skill_hits)

        return (
            "Yes",
            category,
            f"skill_whitelist:{','.join(sorted(skill_hits))[:80]}"
        )

    # 5. Everything else
    return (
        "No",
        "Non-Tech",
        "no_tech_signal"
    )

print("Classification function ready.")


Classification function ready.


In [ ]:
# APPLY TECH CLASSIFICATION

results = df.apply(
    classify,
    axis=1,
    result_type="expand"
)

results.columns = [
    "is_tech_job",
    "tech_role_category",
    "classification_reason"
]

df = pd.concat(
    [df, results],
    axis=1
)


# CHECK RESULTS

print("Tech Job Classification:")
print(df["is_tech_job"].value_counts())

print("\nTech Role Categories:")
print(
    df.loc[df["is_tech_job"] == "Yes", "tech_role_category"]
      .value_counts()
)

print("\nTotal rows:", len(df))

Tech Job Classification:
is_tech_job
No            72342
Yes           22325
Borderline     3015
Name: count, dtype: int64

Tech Role Categories:
tech_role_category
Software Development            8579
Cloud / DevOps                  3419
Data Engineering                2491
QA / Testing                    1849
Data Science / ML               1347
Data Analytics                  1183
Other Tech                      1104
Web Development                  825
IT Support                       638
Cybersecurity                    313
AI / GenAI                       292
Business / Product Analytics     285
Name: count, dtype: int64

Total rows: 97682


In [ ]:
# 8. CLASSIFICATION VALIDATION

print("Total jobs:", len(df))

print("\nTech / Non-Tech:")
print(df["is_tech_job"].value_counts())

print("\nTech role categories:")
print(
    df[df["is_tech_job"] == "Yes"]
    ["tech_role_category"]
    .value_counts()
)

print("\nSample classified jobs:")
print(
    df[
        [
            "title",
            "tagsAndSkills",
            "is_tech_job",
            "tech_role_category"
        ]
    ].head(20)
)

Total jobs: 97682

Tech / Non-Tech:
is_tech_job
No            72342
Yes           22325
Borderline     3015
Name: count, dtype: int64

Tech role categories:
tech_role_category
Software Development            8579
Cloud / DevOps                  3419
Data Engineering                2491
QA / Testing                    1849
Data Science / ML               1347
Data Analytics                  1183
Other Tech                      1104
Web Development                  825
IT Support                       638
Cybersecurity                    313
AI / GenAI                       292
Business / Product Analytics     285
Name: count, dtype: int64

Sample classified jobs:
                                                                     title  \
0                                                Sr. HR Recruiter (NON IT)   
1                                                  Fire And Safety Officer   
2                              Opening For Performance Marketing - Chennai   
3                

In [ ]:
# 9. CHECK BORDERLINE JOBS

print("Borderline jobs:")
print(
    df[df["is_tech_job"] == "Borderline"][
        [
            "title",
            "tagsAndSkills",
            "tech_role_category"
        ]
    ].head(30)
)

Borderline jobs:
                                                                     title  \
133                                                       Business Analyst   
296                                                          Mis Executive   
374                                        Urgent Opening For SAP ABAP +PP   
397                                           Sap C4C Technical Consultant   
447                                                      Sap Sd Consultant   
496                       VP Business Analyst - Lending : Chennai 'Urgent'   
561                                                     SAP ABAP Developer   
813                                               Tririga Business Analyst   
963                                     Corporate Trainer  - SAP MDG & GRC   
965                                                  SAP Fresher - Chennai   
1153                                                  Erp Support Engineer   
1270                                           

In [ ]:
# 10. TECH + FRESHER-FRIENDLY JOBS

fresher_tech = df[
    (df["is_tech_job"] == "Yes") &
    (df["is_fresher_friendly"] == 1)
].copy()

print("Total tech jobs:",
      (df["is_tech_job"] == "Yes").sum())

print("Tech jobs friendly for freshers:",
      len(fresher_tech))

print("\nTech role categories for fresher-friendly jobs:")

print(
    fresher_tech["tech_role_category"]
    .value_counts()
)

print("\nSample fresher-friendly tech jobs:")

print(
    fresher_tech[
        [
            "title",
            "companyName",
            "city",
            "tech_role_category",
            "minimum_experience_clean",
            "maximum_experience_clean"
        ]
    ].head(20)
)

Total tech jobs: 22325
Tech jobs friendly for freshers: 1803

Tech role categories for fresher-friendly jobs:
tech_role_category
Software Development            549
IT Support                      281
QA / Testing                    172
Data Analytics                  154
Web Development                 153
Cloud / DevOps                  147
Data Science / ML               102
Data Engineering                 76
Business / Product Analytics     74
Other Tech                       47
Cybersecurity                    29
AI / GenAI                       19
Name: count, dtype: int64

Sample fresher-friendly tech jobs:
                                                                    title  \
24        4LPA CTC For Email Chat Backend & Non Voice Process For Kolkata   
159                          Hiring For MIS Executive- US Staffing @Noida   
236                                     Associate Mako Product Specialist   
249                                                        Data Scien

In [ ]:
# 11. CREATE SQL-READY DATASET

sql_df = df.copy()

print("SQL-ready dataset shape:", sql_df.shape)
print("Total rows:", len(sql_df))

SQL-ready dataset shape: (97682, 41)
Total rows: 97682


In [ ]:
# 11. EXPORT CLEANED DATA FOR POSTGRESQL

df.to_csv("india_tech_market_cleaned.csv", index=False)

print("CSV exported successfully!")
print("Rows:", len(df))
print("Columns:", len(df.columns))

CSV exported successfully!
Rows: 97682
Columns: 41


In [ ]:
from google.colab import files

files.download("india_tech_market_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>